In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
PATH = "/kaggle/input/competitions/ieee-fraud-detection"
id = pd.read_csv(PATH + "/train_identity.csv")
tr = pd.read_csv(PATH+"/train_transaction.csv")

In [3]:
!pip install mlflow dagshub xgboost -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 88.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 70.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 44.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
import xgboost as xgb
import mlflow.sklearn
import mlflow
import dagshub

In [5]:
TARGET = "isFraud"
TR_MISS_TH = 0.8
ID_MISS_TH = 0.7
CARDINALITY_TH = 20
CORRELATION_TH = 0.85

In [6]:
dagshub.init(repo_owner='lukaLomadze', repo_name='ML_Assignment_2', mlflow=True)
mlflow.set_tracking_uri("https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow")
mlflow.set_experiment('XGBoost_better')

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=b0e44495-a0fd-4bf4-9fa0-bb0ca64f9008&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=8e81a3cf12fe028d31efa1f17d792f1756709a7aa7cce5adca055278684e802e




Accessing as lukaLomadze

Initialized MLflow to track repo "lukaLomadze/ML_Assignment_2"

Repository lukaLomadze/ML_Assignment_2 initialized!

2026/05/04 19:46:18 INFO mlflow.tracking.fluent: Experiment with name 'XGBoost_better' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/ac9fdf1ca4a343669d27ed6efed1ea21', creation_time=1777923978138, experiment_id='6', last_update_time=1777923978138, lifecycle_stage='active', name='XGBoost_better', tags={}, trace_location=None, workspace='default'>

# Cleaning

In [7]:
class DropHighMissing(BaseEstimator, TransformerMixin):
    def __init__(self, tr_th=TR_MISS_TH, id_th=ID_MISS_TH, identity_cols=None):
        self.tr_th = tr_th
        self.id_th = id_th
        self.identity_cols = identity_cols

    def fit(self, X, y=None):
        id_set = set(self.identity_cols) if self.identity_cols else set()
        keep = []
        for col in X.columns:
            miss_rate = X[col].isnull().mean()
            threshold = self.id_th if col in id_set else self.tr_th
            if miss_rate <= threshold:
                keep.append(col)
        self.cols_ = keep
        return self

    def transform(self, X):
        return X[self.cols_]

In [8]:
train = tr.merge(id, on="TransactionID", how="left")
y = train[TARGET].copy()

print(f"Train shape: {train.shape}")
print(f"Fraud rate: {y.mean()}")

Train shape: (590540, 434)
Fraud rate: 0.03499000914417313


In [9]:
cat_cols = train.select_dtypes(include='object').columns.tolist()
num_cols = train.select_dtypes(exclude='object').columns.tolist()

In [10]:
with mlflow.start_run(run_name="xgboost_cleaning"):
    mlflow.log_param("TR_MISS_TH", TR_MISS_TH)
    mlflow.log_param("ID_MISS_TH", ID_MISS_TH)
    mlflow.log_param("Final shape", f"{train.shape[0]} X {train.shape[1]}")
    mlflow.log_param("Num cols", len(num_cols))
    mlflow.log_param("Cat cols", len(cat_cols))
    mlflow.log_param("Fraud rate", float(y.mean()))

🏃 View run xgboost_cleaning at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/6/runs/869ddea9b38e4402ac64ae6fd3c2e982
🧪 View experiment at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/6


# Feature Engineering

In [11]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, identity_cols=None, identity_ids=None, target=None):
        self.identity_cols = identity_cols
        self.identity_ids = identity_ids
        self.target = target

    def fit(self, X, y=None):
        uid_cols = ["card1", "addr1"]
        if all(c in X.columns for c in uid_cols) and "TransactionAmt" in X.columns:
            uid = X["card1"].astype(str) + "_" + X["addr1"].astype(str)
            tmp = X["TransactionAmt"].copy()
            tmp.index = uid
            self.uid_stats_ = tmp.groupby(level=0).agg(
                uid_tx_count="count",
                uid_amt_mean="mean",
                uid_amt_std="std"
            )
        else:
            self.uid_stats_ = None

        freq_cols = ["card1", "card2", "card3", "card5", "addr1", "addr2"]
        self.freq_maps_ = {}
        for col in freq_cols:
            if col in X.columns:
                self.freq_maps_[col] = X[col].astype(str).value_counts(normalize=True)

        return self

    def transform(self, X):
        X = X.copy()

        if "TransactionAmt" in X.columns:
            X["TransactionAmt_log"]   = np.log1p(X["TransactionAmt"])
            X["TransactionAmt_round"] = np.round(X["TransactionAmt"]).astype("float32")

        if "TransactionDT" in X.columns:
            X["TransactionHour"]  = (X["TransactionDT"] // 3600) % 24
            X["TransactionDay"]   = (X["TransactionDT"] // 86400) % 7
            X["TransactionWeek"]  = (X["TransactionDT"] // 86400) // 7
            X["TransactionMonth"] = (X["TransactionDT"] // 86400) // 30

        if self.uid_stats_ is not None:
            uid = X["card1"].astype(str) + "_" + X["addr1"].astype(str)
            X["uid_tx_count"] = uid.map(self.uid_stats_["uid_tx_count"]).astype("float32")
            X["uid_amt_mean"] = uid.map(self.uid_stats_["uid_amt_mean"]).astype("float32")
            X["uid_amt_std"]  = uid.map(self.uid_stats_["uid_amt_std"]).astype("float32")

        for col, freq_map in self.freq_maps_.items():
            if col in X.columns:
                X[f"{col}_freq"] = X[col].astype(str).map(freq_map).fillna(0).astype("float32")

       
        if self.identity_ids is not None and "TransactionID" in X.columns:
            X["has_identity"] = X["TransactionID"].isin(self.identity_ids).astype(int)

        drop_cols = []
        if "TransactionID" in X.columns:
            drop_cols.append("TransactionID")
        if self.target and self.target in X.columns:
            drop_cols.append(self.target)
        if drop_cols:
            X = X.drop(columns=drop_cols)

        return X

In [12]:
low_card_cols = [c for c in cat_cols if train[c].nunique() <= CARDINALITY_TH]
high_card_cols = [c for c in cat_cols if train[c].nunique() > CARDINALITY_TH]

print(f"Low cardinality cols: {len(low_card_cols)}")
print(f"High cardinality cols: {len(high_card_cols)}")

Low cardinality cols: 25
High cardinality cols: 6


In [13]:
with mlflow.start_run(run_name="xgboost_feature_engineering"):
    mlflow.log_param("New features", "TransactionDay, TransactionHour, TransactionWeek, TransactionMonth, TransactionAmt_log, TransactionAmt_round, has_identity, uid_tx_count, uid_amt_mean, uid_amt_std, card*_freq, addr*_freq")
    mlflow.log_param("CARDINALITY_TH", CARDINALITY_TH)
    mlflow.log_param("Low cardinality cols", len(low_card_cols))
    mlflow.log_param("High cardinality cols", len(high_card_cols))

🏃 View run xgboost_feature_engineering at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/6/runs/6058fe210e28463c8d3bbfa6af0e04b4
🧪 View experiment at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/6


# Feature Selection

In [14]:
class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, th=CORRELATION_TH):
        self.th = th

    def fit(self, X, y=None):
        num = X.select_dtypes(exclude="object")
        if num.shape[1] > 0:
            corr = num.fillna(num.median()).corr().abs()
            upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
            self.drop_cols_ = [c for c in upper.columns if any(upper[c] > self.th)]
        else:
            self.drop_cols_ = []
        return self

    def transform(self, X):
        return X.drop(columns=self.drop_cols_, errors="ignore")

In [15]:
# with mlflow.start_run(run_name="xgboost_feature_selection"):
#     mlflow.log_param("Method", "correlation_filter")
#     mlflow.log_param("CORRELATION_TH", CORRELATION_TH)

# Preprocessor

In [16]:
class Preprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, cardinality_th=20):
        self.cardinality_th = cardinality_th

    def _build(self, X):
        cat_cols  = X.select_dtypes(include="object").columns.tolist()
        low_card  = [c for c in cat_cols if X[c].nunique() <= self.cardinality_th]
        high_card = [c for c in cat_cols if X[c].nunique() >  self.cardinality_th]

        ohe_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ])
        
        ord_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ])

        
        transformers = []
        if low_card:
            transformers.append(("ohe", ohe_pipe, low_card))
        if high_card:
            transformers.append(("ord", ord_pipe, high_card))

        return ColumnTransformer(transformers, remainder="passthrough")

    def fit(self, X, y=None):
        self.ct_ = self._build(X)
        self.ct_.fit(X, y)
        return self

    def transform(self, X):
        return self.ct_.transform(X)

In [17]:
def build_pipeline(n_estimators, max_depth, learning_rate, subsample,
                   colsample_bytree, min_child_weight, gamma,
                   scale_pos_weight, colsample_bylevel=1.0):
 

    identity_cols = [c for c in id.columns if c != "TransactionID"]
    identity_ids  = set(id["TransactionID"].unique())

    xgb_params = {
        "n_estimators":      n_estimators,
        "max_depth":         max_depth,
        "learning_rate":     learning_rate,
        "subsample":         subsample,
        "colsample_bytree":  colsample_bytree,
        "colsample_bylevel": colsample_bylevel,
        "min_child_weight":  min_child_weight,
        "gamma":             gamma,
        "scale_pos_weight":  scale_pos_weight,
        "eval_metric":       "auc",
        "tree_method":       "hist",   
        "random_state":      42,
        "n_jobs":            -1,
    }

    return Pipeline([
        ("drop",  DropHighMissing(tr_th=TR_MISS_TH, id_th=ID_MISS_TH, identity_cols=identity_cols)),
        ("feat",  FeatureEngineer(identity_cols=identity_cols, identity_ids=identity_ids, target=TARGET)),
        ("prep",  Preprocessor(cardinality_th=CARDINALITY_TH)),
        ("model", xgb.XGBClassifier(**xgb_params))
    ])

# Training

In [18]:
scale_pos_weight_value = float((y == 0).sum() / (y == 1).sum())

In [ ]:
spw = scale_pos_weight_value

PARAM_GRID = [
    
    dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, colsample_bylevel=1.0, min_child_weight=1,  gamma=0,   scale_pos_weight=spw),
    dict(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, colsample_bylevel=1.0, min_child_weight=1,  gamma=0,   scale_pos_weight=spw),
    dict(n_estimators=500, max_depth=4, learning_rate=0.02, subsample=0.8, colsample_bytree=0.8, colsample_bylevel=1.0, min_child_weight=1,  gamma=0,   scale_pos_weight=spw),
    dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, colsample_bylevel=1.0, min_child_weight=10, gamma=0,   scale_pos_weight=spw),
    dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, colsample_bylevel=1.0, min_child_weight=1,  gamma=1,   scale_pos_weight=spw),
    dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.7, colsample_bytree=0.6, colsample_bylevel=0.8, min_child_weight=5,  gamma=0.5, scale_pos_weight=spw),
    dict(n_estimators=700, max_depth=5, learning_rate=0.02, subsample=0.8, colsample_bytree=0.7, colsample_bylevel=1.0, min_child_weight=5,  gamma=0,   scale_pos_weight=spw),
]

In [20]:

CV = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

best_auc    = -1
best_params = None
results     = []

X = train

In [21]:
for p in PARAM_GRID:
    with mlflow.start_run(run_name=f"XGB_n={p['n_estimators']}_d={p['max_depth']}_lr={p['learning_rate']}_mcw={p['min_child_weight']}_g={p['gamma']}"):

        pipe = build_pipeline(
            n_estimators      = p["n_estimators"],
            max_depth         = p["max_depth"],
            learning_rate     = p["learning_rate"],
            subsample         = p["subsample"],
            colsample_bytree  = p["colsample_bytree"],
            colsample_bylevel = p.get("colsample_bylevel", 1.0),
            min_child_weight  = p["min_child_weight"],
            gamma             = p["gamma"],
            scale_pos_weight  = p["scale_pos_weight"],
        )

        scores   = cross_val_score(pipe, X, y, cv=CV, scoring="roc_auc", n_jobs=1)
        mean_auc = scores.mean()
        std_auc  = scores.std()

        mlflow.log_params(p)
        mlflow.log_metric("cv_auc_mean", mean_auc)
        mlflow.log_metric("cv_auc_std",  std_auc)

        results.append((p, mean_auc))

        if mean_auc > best_auc:
            best_auc    = mean_auc
            best_params = p

        print(f"AUC: {mean_auc} (+/- {std_auc}) | {p}")

AUC: 0.9173478390342703 (+/- 0.0024545156908745316) | {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'colsample_bylevel': 1.0, 'min_child_weight': 1, 'gamma': 0, 'scale_pos_weight': 27.579586700866283}
🏃 View run XGB_n=300_d=4_lr=0.05_mcw=1_g=0 at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/6/runs/39a62a3d350f4a25934b79ab0f866b9b
🧪 View experiment at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/6
AUC: 0.9415478717040551 (+/- 0.0013204592523716203) | {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'colsample_bylevel': 1.0, 'min_child_weight': 1, 'gamma': 0, 'scale_pos_weight': 27.579586700866283}
🏃 View run XGB_n=300_d=6_lr=0.05_mcw=1_g=0 at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/6/runs/ed1a460674784ee0b7f16f91de5f6f0c
🧪 View experiment at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/

In [22]:
print(f"\nBest params: {best_params}  |  CV AUC: {best_auc}")

best_pipe = build_pipeline(
    n_estimators      = best_params["n_estimators"],
    max_depth         = best_params["max_depth"],
    learning_rate     = best_params["learning_rate"],
    subsample         = best_params["subsample"],
    colsample_bytree  = best_params["colsample_bytree"],
    colsample_bylevel = best_params.get("colsample_bylevel", 1.0),
    min_child_weight  = best_params["min_child_weight"],
    gamma             = best_params["gamma"],
    scale_pos_weight  = best_params["scale_pos_weight"],
)

with mlflow.start_run(run_name="XGB_better_best_registered") as run:
    mlflow.log_params(best_params)
    mlflow.log_metric("cv_auc_mean", best_auc)

    best_pipe.fit(X, y)

    model_info = mlflow.sklearn.log_model(
        sk_model              = best_pipe,
        artifact_path         = "model",
        registered_model_name = "XGB_better_Best",
    )

    print(f"Model registered: {model_info.model_uri}")
    print(f"Run ID: {run.info.run_id}")


Best params: {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'colsample_bylevel': 1.0, 'min_child_weight': 1, 'gamma': 0, 'scale_pos_weight': 27.579586700866283}  |  CV AUC: 0.9415478717040551


2026/05/04 20:13:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 20:13:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'XGB_better_Best'.
2026/05/04 20:13:59 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB_better_Best, version 1
Created version '1' of model 'XGB_better_Best'.


Model registered: models:/m-a90ddee14bf0408dacb3283c19086a53
Run ID: 71b003e7258e48449e5b6976185e32b0
🏃 View run XGB_better_best_registered at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/6/runs/71b003e7258e48449e5b6976185e32b0
🧪 View experiment at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/6
